# **Replication of Table 3** 
### $\Sigma$-adjusted method

In [1]:
import cvxpy as cp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy.optimize import minimize
import math
from scipy.stats import skew
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['font.sans-serif'] = ['SimHei']  # 中文黑体
plt.rcParams['axes.unicode_minus'] = False    # 正常显示负号

In [2]:
# 导入CRSP数据
mom = pd.read_csv('E:/Replications/2025_Fall/Baule_2019_Markowitz with regret/F-F_Momentum_Factor_daily.csv')
ff3 = pd.read_csv('E:/Replications/2025_Fall/Baule_2019_Markowitz with regret/F-F_Research_Data_Factors_daily.csv')
ff3["date"] = pd.to_datetime(ff3["date"], format="%Y%m%d")
mom["date"] = pd.to_datetime(mom["date"], format="%Y%m%d")

In [3]:
ff4 = pd.merge(ff3, mom, on="date", how="inner")
start_date = "1926-11-01"
end_date   = "2016-12-31"
ff4 = ff4[(ff4["date"] >= start_date) & (ff4["date"] <= end_date)]
ff4

,date,Mkt-RF,SMB,HML,RF,MOM
0,1926-11-03,0.20,-0.20,-0.33,0.01,0.54
1,1926-11-04,0.59,-0.12,0.65,0.01,-0.51
2,1926-11-05,0.07,-0.11,0.26,0.01,1.17
3,1926-11-06,0.15,-0.29,0.05,0.01,-0.03
4,1926-11-08,0.52,-0.12,0.18,0.01,-0.02
...,...,...,...,...,...,...
23783,2016-12-23,0.19,0.57,-0.50,0.00,-0.39
23784,2016-12-27,0.27,0.22,0.13,0.00,0.28
23785,2016-12-28,-0.87,-0.27,0.08,0.00,-0.13
23786,2016-12-29,-0.04,0.15,-0.33,0.00,0.09


In [4]:
# 选择用于 Table 3 的四个风格列（已为超额收益）
assets = ['Mkt-RF', 'SMB', 'HML', 'MOM']
df = ff4.copy()

# 确保按日期排序且去掉含 NaN 的行
df = df.sort_values('date').reset_index(drop=True)
df_assets = df[assets].astype(float).dropna()
df_assets = df_assets/100
df_assets

,Mkt-RF,SMB,HML,MOM
0,0.0020,-0.0020,-0.0033,0.0054
1,0.0059,-0.0012,0.0065,-0.0051
2,0.0007,-0.0011,0.0026,0.0117
3,0.0015,-0.0029,0.0005,-0.0003
4,0.0052,-0.0012,0.0018,-0.0002
...,...,...,...,...
23783,0.0019,0.0057,-0.0050,-0.0039
23784,0.0027,0.0022,0.0013,0.0028
23785,-0.0087,-0.0027,0.0008,-0.0013
23786,-0.0004,0.0015,-0.0033,0.0009


In [5]:
# 计算每个资产的基本统计量
annual_factor = 252
assets = df_assets.columns
stats_calculations = [('Mean Return', df_assets.mean() * annual_factor), ('Std. Deviation', df_assets.std() * np.sqrt(annual_factor)),
                      ('Skewness', df_assets.skew())]

stats_data = []
index = []

for stat_name, values in stats_calculations:
    stats_data.append(values.round(4))
    index.append(stat_name)
    deviations = values - values.mean()
    stats_data.append(pd.Series([f"({dev:.4f})" for dev in deviations], index=assets))
    index.append('')

panel_a_stats = pd.DataFrame(stats_data, index=index, columns=assets)
panel_a_stats

,Mkt-RF,SMB,HML,MOM
Mean Return,0.0726,0.0136,0.0437,0.0659
,(0.0236),(-0.0353),(-0.0052),(0.0169)
Std. Deviation,0.1694,0.093,0.0933,0.1183
,(0.0509),(-0.0255),(-0.0252),(-0.0002)
Skewness,-0.125,-0.7604,0.7828,-1.6382
,(0.3102),(-0.3252),(1.2180),(-1.2030)


In [6]:
annual_factor = 252
R = df_assets.values        # shape (T, 4)
asset_names = df_assets.columns.tolist()
T, N = R.shape
alpha = 1.0
gamma = 3.0
R_max = R.max(axis=1)       # shape (T,)

In [7]:
def deviation(x):
    return x - x.mean()

In [8]:
# 协方差矩阵 [R, R_max]
ret_ra = np.array([np.cov(R[:, i], R_max, ddof=1)[0, 1] for i in range(N)]) * annual_factor
ret_ra

array([8.21693593e-03, 9.21650649e-05, 2.37090662e-03, 7.64397120e-04])

In [9]:
Z_ret = R - alpha * R_max[:, None]
std_regret_ret = Z_ret.std(axis=0, ddof=1) * np.sqrt(annual_factor)
std_regret_ret

array([0.15784962, 0.1453119 , 0.12885607, 0.15851081])

In [10]:
panelA = pd.DataFrame({'Regret adjustment μ': ret_ra, 'Regret-adjusted std': std_regret_ret},index=asset_names)
panelA_dev = panelA.apply(deviation)
panelA_dev

,Regret adjustment μ,Regret-adjusted std
Mkt-RF,0.005356,0.010218
SMB,-0.002769,-0.002320
HML,-0.000490,-0.018776
MOM,-0.002097,0.010879


In [11]:
panelA_table3 = pd.DataFrame(
    np.vstack([panelA['Regret adjustment μ'].values, panelA_dev['Regret adjustment μ'].values, 
               panelA['Regret-adjusted std'].values, panelA_dev['Regret-adjusted std'].values]),
    index=['Ret-adj μ', '(deviation)', 'Ret-adj std', '(deviation)'],columns=asset_names)

panelA_table3.round(4)

,Mkt-RF,SMB,HML,MOM
Ret-adj μ,0.0082,0.0001,0.0024,0.0008
(deviation),0.0054,-0.0028,-0.0005,-0.0021
Ret-adj std,0.1578,0.1453,0.1289,0.1585
(deviation),0.0102,-0.0023,-0.0188,0.0109


### **preference regret**

In [12]:
var_assets = np.var(R, axis=0, ddof=1)    # shape (N,)
pi = R - gamma * var_assets[None, :]      # (T, N)
# 事后最优pi
pi_max = pi.max(axis=1)                   # (T,)

In [13]:
pref_ra = np.array([np.cov(R[:, i], pi_max, ddof=1)[0, 1]for i in range(N)]) * annual_factor
pref_ra

array([0.00806343, 0.00012582, 0.00238443, 0.00076689])

In [14]:
Z_pref = R - alpha * pi_max[:, None]
std_regret_pref = Z_pref.std(axis=0, ddof=1) * np.sqrt(annual_factor)
std_regret_pref

array([0.15852302, 0.14475592, 0.12838566, 0.15819841])

In [15]:
panelA_pref = pd.DataFrame({'Regret adjustment μ': pref_ra,'Regret-adjusted std': std_regret_pref},index=asset_names)
panelA_pref_dev = panelA_pref.apply(deviation)

In [16]:
panelA_pref_table3 = pd.DataFrame(
    np.vstack([panelA_pref['Regret adjustment μ'].values, panelA_pref_dev['Regret adjustment μ'].values,
               panelA_pref['Regret-adjusted std'].values, panelA_pref_dev['Regret-adjusted std'].values]),
    index=['Pref-adj μ (u–σ)', '(deviation)', 'Pref-adj std (u–σ)', '(deviation)'],
    columns=asset_names)
panelA_pref_table3.round(4)

,Mkt-RF,SMB,HML,MOM
Pref-adj μ (u–σ),0.0081,0.0001,0.0024,0.0008
(deviation),0.0052,-0.0027,-0.0005,-0.0021
Pref-adj std (u–σ),0.1585,0.1448,0.1284,0.1582
(deviation),0.0111,-0.0027,-0.0191,0.0107


In [17]:
gamma_small = 1e-8

var_assets = np.var(R, axis=0, ddof=1)
pi_small = R - gamma_small * var_assets[None, :]
pi_max_small = pi_small.max(axis=1)

# preference → return
np.allclose(pi_max_small, R.max(axis=1), atol=1e-6)

True

#### **Min-Var**

In [18]:
# 协方差矩阵
Sigma = np.cov(R, rowvar=False, ddof=1)

# Min-Var 权重
ones = np.ones(N)
Sigma_inv = np.linalg.inv(Sigma)
w_mv = Sigma_inv @ ones
w_mv = w_mv / (ones @ Sigma_inv @ ones)

# Min-Var 基准收益
R_mv = R @ w_mv    # shape (T,)

In [19]:
pref_ra_minvar = np.array([np.cov(R[:, i], R_mv, ddof=1)[0, 1] for i in range(N)]) * annual_factor
pref_ra_minvar

array([0.00251059, 0.00251059, 0.00251059, 0.00251059])

In [20]:
Z_pref_mv = R - alpha * R_mv[:, None]
std_regret_pref_mv = Z_pref_mv.std(axis=0, ddof=1) * np.sqrt(annual_factor)
std_regret_pref_mv

array([0.16183938, 0.07836765, 0.07865945, 0.10721978])

In [21]:
panelA_pref_mv = pd.DataFrame({'Regret adjustment μ': pref_ra_minvar, 'Regret-adjusted std': std_regret_pref_mv}, index=asset_names)
panelA_pref_mv_dev = panelA_pref_mv.apply(deviation)

In [22]:
panelA_pref_mv_table3 = pd.DataFrame(
    np.vstack([panelA_pref_mv['Regret adjustment μ'].values, panelA_pref_mv_dev['Regret adjustment μ'].values,
               panelA_pref_mv['Regret-adjusted std'].values, panelA_pref_mv_dev['Regret-adjusted std'].values]),
    index=['Pref-adj μ (Min-Var)','(deviation)', 'Pref-adj std (Min-Var)', '(deviation)'],columns=asset_names)

panelA_pref_mv_table3.round(4)

,Mkt-RF,SMB,HML,MOM
Pref-adj μ (Min-Var),0.0025,0.0025,0.0025,0.0025
(deviation),0.0000,0.0000,-0.0000,-0.0000
Pref-adj std (Min-Var),0.1618,0.0784,0.0787,0.1072
(deviation),0.0553,-0.0282,-0.0279,0.0007


### **Panel A 汇总**

In [23]:
panelA_table3 = panelA_table3.round(4)
panelA_pref_table3 = panelA_pref_table3.round(4)
panelA_pref_mv_table3 = panelA_pref_mv_table3.round(4)
df_all = (pd.concat([panel_a_stats, panelA_table3, panelA_pref_table3, panelA_pref_mv_table3],axis=0, ignore_index=False))
df_all.round(4)

,Mkt-RF,SMB,HML,MOM
Mean Return,0.0726,0.0136,0.0437,0.0659
,(0.0236),(-0.0353),(-0.0052),(0.0169)
Std. Deviation,0.1694,0.093,0.0933,0.1183
,(0.0509),(-0.0255),(-0.0252),(-0.0002)
Skewness,-0.125,-0.7604,0.7828,-1.6382
,(0.3102),(-0.3252),(1.2180),(-1.2030)
Ret-adj μ,0.0082,0.0001,0.0024,0.0008
(deviation),0.0054,-0.0028,-0.0005,-0.0021
Ret-adj std,0.1578,0.1453,0.1289,0.1585
(deviation),0.0102,-0.0023,-0.0188,0.0109


### **Panel B**

In [24]:
mu = R.mean(axis=0) * annual_factor               # (4,)
Sigma = np.cov(R, rowvar=False, ddof=1) * annual_factor  # (4,4)

In [25]:
# Markowitz
N = R.shape[1]
w = cp.Variable(N)
objective = cp.Maximize(mu @ w - 0.5 * gamma * cp.quad_form(w, Sigma))

constraints = [cp.sum(w) == 1,w >= 0]
problem = cp.Problem(objective, constraints)
problem.solve()
w_markowitz = w.value
w_markowitz

array([3.94054261e-01, 1.03446613e-22, 1.25120244e-22, 6.05945739e-01])

In [26]:
dev_w_markowitz = deviation(w_markowitz)
panelB_markowit = pd.DataFrame(np.vstack([w_markowitz, dev_w_markowitz]),
    index=['Markowitz (μ–σ)','(deviation)'],columns=asset_names)
panelB_markowitz = panelB_markowit.round(4)
panelB_markowitz

,Mkt-RF,SMB,HML,MOM
Markowitz (μ–σ),0.3941,0.00,0.00,0.6059
(deviation),0.1441,-0.25,-0.25,0.3559


In [27]:
# Min-Var
Sigma = np.cov(R, rowvar=False, ddof=1) * annual_factor
N = R.shape[1]
w = cp.Variable(N)
objective = cp.Minimize(cp.quad_form(w, Sigma))
constraints = [cp.sum(w) == 1,w >= 0]

# 求解
problem = cp.Problem(objective, constraints)
problem.solve()
# 提取权重
w_minvar = w.value
w_minvar

array([0.10985191, 0.33797708, 0.32441039, 0.22776061])

In [28]:
dev_w_minvar = deviation(w_minvar)
panelB_w_minvar = pd.DataFrame(np.vstack([w_minvar, dev_w_minvar]),
    index=['Markowitz (Min-Var)','(deviation)'],columns=asset_names)
panelB_minvar = panelB_w_minvar.round(4)
panelB_minvar

,Mkt-RF,SMB,HML,MOM
Markowitz (Min-Var),0.1099,0.338,0.3244,0.2278
(deviation),-0.1401,0.088,0.0744,-0.0222


In [29]:
# Pure regret · return view · μ–σ (γ=3)
R_max = R.max(axis=1)
Z_ret = R - R_max[:, None]
Sigma_ra = np.cov(Z_ret, rowvar=False, ddof=1) * annual_factor

In [30]:
objective = cp.Maximize(mu @ w - 0.5 * gamma * cp.quad_form(w, Sigma_ra))
constraints = [cp.sum(w) == 1,w >= 0]
problem = cp.Problem(objective, constraints)
problem.solve()
w_pure_ret_mu_sigma = w.value
w_pure_ret_mu_sigma

array([ 5.48464781e-01, -1.32553829e-23,  1.09675033e-22,  4.51535219e-01])

In [31]:
# 非负
(w_pure_ret_mu_sigma >= -1e-10).all()

np.True_

In [32]:
dev_w_pure_ret_mu_sigma = deviation(w_pure_ret_mu_sigma)
panelB_pure_ret_mu_sigma = pd.DataFrame(np.vstack([w_pure_ret_mu_sigma, dev_w_pure_ret_mu_sigma]),
                                        index=['Ret regret(μ–σ)','(deviation)'], columns=asset_names)
panelB_ret_regret_mu_sigma = panelB_pure_ret_mu_sigma.round(4)
panelB_ret_regret_mu_sigma

,Mkt-RF,SMB,HML,MOM
Ret regret(μ–σ),0.5485,-0.00,0.00,0.4515
(deviation),0.2985,-0.25,-0.25,0.2015


In [33]:
# Pure regret · return view  Min Var
objective = cp.Minimize(cp.quad_form(w, Sigma_ra))
constraints = [cp.sum(w) == 1,w >= 0]
problem = cp.Problem(objective, constraints)
problem.solve()

w_pure_ret_minvar = w.value
w_pure_ret_minvar

array([0.31665183, 0.19916653, 0.29322687, 0.19095477])

In [34]:
dev_pure_ret_minvar = deviation(w_pure_ret_minvar)
panelB_pure_ret_minvar = pd.DataFrame(np.vstack([w_pure_ret_minvar, dev_pure_ret_minvar]),
                                      index=['Ret regret(Min-Var)', '(deviation)'], columns=asset_names)

panelB_Ret_regret_minvar = panelB_pure_ret_minvar.round(4)
panelB_Ret_regret_minvar

,Mkt-RF,SMB,HML,MOM
Ret regret(Min-Var),0.3167,0.1992,0.2932,0.191
(deviation),0.0667,-0.0508,0.0432,-0.059


In [35]:
# Pure regret · pref view · μ–σ (γ=3)
gamma = 3.0
var_assets = np.var(R, axis=0, ddof=1)
pi = R - gamma * var_assets[None, :]
pi_max = pi.max(axis=1)
Z_pref = R - pi_max[:, None]
Sigma_pref = np.cov(Z_pref, rowvar=False, ddof=1) * annual_factor
Sigma_pref_ra = Sigma + Sigma_pref 

In [36]:
w = cp.Variable(N)
objective = cp.Maximize(mu @ w - 0.5 * gamma * cp.quad_form(w, Sigma_pref_ra))
constraints = [cp.sum(w) == 1, w >= 0]
problem = cp.Problem(objective, constraints)
problem.solve()

w_pure_pref_mu_sigma = w.value
w_pure_pref_mu_sigma

array([3.59015091e-01, 4.00292028e-23, 1.96991456e-01, 4.43993452e-01])

In [37]:
dev_pure_pref_mu_sigma = deviation(w_pure_pref_mu_sigma)
panelB_pure_pref_mu_sigma = pd.DataFrame(np.vstack([w_pure_pref_mu_sigma, dev_pure_pref_mu_sigma]),
                                      index=['Pref regret(μ–σ)', '(deviation)'], columns=asset_names)

panelB_Pref_regret_minvar = panelB_pure_pref_mu_sigma.round(4)
panelB_Pref_regret_minvar

,Mkt-RF,SMB,HML,MOM
Pref regret(μ–σ),0.359,0.00,0.197,0.444
(deviation),0.109,-0.25,-0.053,0.194


#### **Min_Var**

In [43]:
# Cov(R)
Sigma_R = np.cov(R, rowvar=False, ddof=1) * annual_factor

# Var(pi_max)
var_pi_max = np.var(pi_max, ddof=1) * annual_factor

# Cov(R, pi_max)
cov_R_pi = np.array([
    np.cov(R[:, i], pi_max, ddof=1)[0, 1]
    for i in range(N)
]) * annual_factor

# Preference-regret covariance (paper definition)
Sigma_pref = (
    Sigma_R
    + var_pi_max * np.ones((N, N))
    - np.outer(cov_R_pi, np.ones(N))
    - np.outer(np.ones(N), cov_R_pi)
)
Sigma_pref_ra = alpha * Sigma + Sigma_pref

In [44]:
objective = cp.Minimize(cp.quad_form(w, Sigma_pref_ra))
constraints = [cp.sum(w) == 1,w >= 0]
problem = cp.Problem(objective, constraints)
problem.solve()

w_pure_pref_minvar = w.value
w_pure_pref_minvar

array([0.21050258, 0.26980961, 0.31059666, 0.20909116])

In [39]:
dev_pure_pref_minvar = deviation(w_pure_pref_minvar)
panelB_pure_pref_minvar = pd.DataFrame(np.vstack([w_pure_pref_minvar, dev_pure_pref_minvar]),
                                       index=['Pref regret(Min-Var)','(deviation)'],columns=asset_names)

panelB_pure_pref_minvar.round(4)

,Mkt-RF,SMB,HML,MOM
Pref regret(Min-Var),0.2105,0.2698,0.3106,0.2091
(deviation),-0.0395,0.0198,0.0606,-0.0409


In [40]:
df_all = (pd.concat([panelB_markowitz, panelB_minvar, panelB_ret_regret_mu_sigma, panelB_Ret_regret_minvar, 
                     panelB_Pref_regret_minvar, panelB_pure_pref_minvar],axis=0, ignore_index=False))
df_all.round(4)

,Mkt-RF,SMB,HML,MOM
Markowitz (μ–σ),0.3941,0.0000,0.0000,0.6059
(deviation),0.1441,-0.2500,-0.2500,0.3559
Markowitz (Min-Var),0.1099,0.3380,0.3244,0.2278
(deviation),-0.1401,0.0880,0.0744,-0.0222
Ret regret(μ–σ),0.5485,-0.0000,0.0000,0.4515
(deviation),0.2985,-0.2500,-0.2500,0.2015
Ret regret(Min-Var),0.3167,0.1992,0.2932,0.1910
(deviation),0.0667,-0.0508,0.0432,-0.0590
Pref regret(μ–σ),0.3590,0.0000,0.1970,0.4440
(deviation),0.1090,-0.2500,-0.0530,0.1940
